# IAA Calculator — Inter-Annotator Agreement

**Mục tiêu:** Merge 2 file annotation, tính Cohen's Kappa, phân tích conflict.

**Cần chuẩn bị:**
- `annotation_A.xlsx` — file Annotator A đã điền nhãn
- `annotation_B.xlsx` — file Annotator B đã điền nhãn

Cả 2 file phải có cùng cấu trúc với file gốc `annotation_pairs.xlsx`, cột nhãn tên là `NHÃN (exact/partial/irrelevant)`.

## Bước 0 — Cài thư viện

In [14]:
!pip install openpyxl scikit-learn pandas -q

## Bước 1 — Upload 2 file annotation

In [15]:
from google.colab import files
FILE_A = '/content/annotation_pairs_Tri.xlsx'
print(f"File A: {FILE_A}")

File A: /content/annotation_pairs_Tri.xlsx


In [16]:

FILE_B = '/content/annotation_pairs_Binh.xlsx'
print(f"File B: {FILE_B}")

File B: /content/annotation_pairs_Binh.xlsx


## Bước 2 — Load và kiểm tra dữ liệu

In [17]:
import pandas as pd
from openpyxl import load_workbook

LABEL_COL  = 'NHÃN (exact/partial/irrelevant)'   # tên cột nhãn
ID_COL     = 'ID'
VALID_LABELS = {'exact', 'partial', 'irrelevant'}

def load_annotation(filepath, annotator_name):
    df = pd.read_excel(filepath, sheet_name='Annotation')
    df.columns = df.columns.str.strip()

    # Normalize nhãn: lowercase, strip khoảng trắng
    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().str.lower()
    df[LABEL_COL] = df[LABEL_COL].replace('none', pd.NA)

    total = len(df)
    labeled = df[LABEL_COL].isin(VALID_LABELS).sum()
    invalid = df[~df[LABEL_COL].isin(VALID_LABELS) & df[LABEL_COL].notna()]

    print(f"\n=== {annotator_name} ===")
    print(f"  Tổng cặp:     {total}")
    print(f"  Đã gán nhãn:  {labeled}")
    print(f"  Bỏ trống:     {total - labeled}")
    if len(invalid) > 0:
        print(f"  ⚠️  Nhãn không hợp lệ ({len(invalid)} dòng): {invalid[LABEL_COL].unique().tolist()}")
        print(f"     → Các nhãn hợp lệ: exact / partial / irrelevant")
    print(f"  Phân bổ nhãn:")
    print(df[LABEL_COL].value_counts().to_string())
    return df

df_A = load_annotation(FILE_A, 'Annotator A')
df_B = load_annotation(FILE_B, 'Annotator B')


=== Annotator A ===
  Tổng cặp:     200
  Đã gán nhãn:  200
  Bỏ trống:     0
  Phân bổ nhãn:
NHÃN (exact/partial/irrelevant)
irrelevant    107
exact          82
partial        11

=== Annotator B ===
  Tổng cặp:     200
  Đã gán nhãn:  200
  Bỏ trống:     0
  Phân bổ nhãn:
NHÃN (exact/partial/irrelevant)
irrelevant    107
exact          77
partial        16


## Bước 3 — Merge 2 file theo pair_id

In [18]:
# Giữ lại các cột cần thiết
df_A_slim = df_A[[ID_COL, 'Query (của user)', 'Tên sản phẩm', 'Category',
                   'Loại', 'Màu', LABEL_COL, 'Ghi chú']].copy()
df_A_slim.columns = [ID_COL, 'query', 'product_name', 'category',
                      'type', 'colors', 'label_A', 'note_A']

df_B_slim = df_B[[ID_COL, LABEL_COL, 'Ghi chú']].copy()
df_B_slim.columns = [ID_COL, 'label_B', 'note_B']

merged = pd.merge(df_A_slim, df_B_slim, on=ID_COL, how='inner')

# Chỉ giữ các dòng cả 2 đều đã gán nhãn hợp lệ
valid_mask = merged['label_A'].isin(VALID_LABELS) & merged['label_B'].isin(VALID_LABELS)
df_valid = merged[valid_mask].copy()
df_skipped = merged[~valid_mask]

print(f"✅ Merge xong: {len(merged)} cặp")
print(f"   Cả 2 đã gán nhãn: {len(df_valid)} cặp")
if len(df_skipped) > 0:
    print(f"   ⚠️  Bỏ qua {len(df_skipped)} cặp (thiếu nhãn 1 trong 2 người)")
    print(df_skipped[[ID_COL, 'label_A', 'label_B']].to_string(index=False))

✅ Merge xong: 200 cặp
   Cả 2 đã gán nhãn: 200 cặp


## Bước 4 — Tính IAA

In [19]:
from sklearn.metrics import cohen_kappa_score, confusion_matrix, classification_report
import numpy as np

labels_A = df_valid['label_A'].tolist()
labels_B = df_valid['label_B'].tolist()
n = len(labels_A)

# Percent Agreement
agree = sum(a == b for a, b in zip(labels_A, labels_B))
pct_agree = agree / n

# Cohen's Kappa
kappa = cohen_kappa_score(labels_A, labels_B)

# Đánh giá kappa
def interpret_kappa(k):
    if k > 0.8:  return "Rất tốt (near-perfect) "
    if k > 0.6:  return "Tốt (substantial) "
    if k > 0.4:  return "Chấp nhận được (moderate)   — cần review conflict"
    if k > 0.2:  return "Yếu (fair)  — cần họp lại, sửa guideline"
    return "Rất yếu  — phải viết lại guideline từ đầu"

print("=" * 55)
print("INTER-ANNOTATOR AGREEMENT")
print("=" * 55)
print(f"Số cặp đánh giá:    {n}")
print(f"Số cặp đồng thuận:  {agree} ({pct_agree:.1%})")
print(f"Percent Agreement:  {pct_agree:.1%}")
print(f"Cohen's Kappa:      {kappa:.3f}")
print(f"Đánh giá:           {interpret_kappa(kappa)}")

# Confusion matrix
label_order = ['exact', 'partial', 'irrelevant']
cm = confusion_matrix(labels_A, labels_B, labels=label_order)

print(f"\n{'':5} Confusion Matrix (hàng = A, cột = B):")
print(f"{'':15} {'exact':>10} {'partial':>10} {'irrelevant':>12}")
for i, label in enumerate(label_order):
    row = cm[i]
    marker = ''
    print(f"A={label:<12} {row[0]:>10} {row[1]:>10} {row[2]:>12} {marker}")

# Highlight các ô conflict lớn nhất
print(f"\n Conflict lớn nhất:")
conflicts = []
for i, la in enumerate(label_order):
    for j, lb in enumerate(label_order):
        if i != j and cm[i][j] > 0:
            conflicts.append((cm[i][j], la, lb))
conflicts.sort(reverse=True)
for count, la, lb in conflicts[:5]:
    pct = count / n * 100
    print(f"   A={la} vs B={lb}: {count} cặp ({pct:.1f}%)")

INTER-ANNOTATOR AGREEMENT
Số cặp đánh giá:    200
Số cặp đồng thuận:  171 (85.5%)
Percent Agreement:  85.5%
Cohen's Kappa:      0.737
Đánh giá:           Tốt (substantial) 

      Confusion Matrix (hàng = A, cột = B):
                     exact    partial   irrelevant
A=exact                67         14            1 
A=partial               5          2            4 
A=irrelevant            5          0          102 

 Conflict lớn nhất:
   A=exact vs B=partial: 14 cặp (7.0%)
   A=partial vs B=exact: 5 cặp (2.5%)
   A=irrelevant vs B=exact: 5 cặp (2.5%)
   A=partial vs B=irrelevant: 4 cặp (2.0%)
   A=exact vs B=irrelevant: 1 cặp (0.5%)


## Bước 5 — Phân tích conflict theo Tier

In [20]:
# Load Pipeline Metadata từ file A (tier info)
try:
    df_meta = pd.read_excel(FILE_A, sheet_name='Pipeline Metadata')
    df_meta.columns = df_meta.columns.str.strip()

    df_valid_meta = pd.merge(df_valid, df_meta[['pair_id', 'tier', 'color_mismatch']],
                              left_on=ID_COL, right_on='pair_id', how='left')

    df_valid_meta['is_conflict'] = df_valid_meta['label_A'] != df_valid_meta['label_B']

    print("=" * 55)
    print("CONFLICT ANALYSIS BY TIER")
    print("=" * 55)
    for tier in ['T1_high', 'T2_edge', 'T3_random']:
        subset = df_valid_meta[df_valid_meta['tier'] == tier]
        if len(subset) == 0:
            continue
        n_conflict = subset['is_conflict'].sum()
        pct = n_conflict / len(subset) * 100
        print(f"\n{tier} ({len(subset)} cặp):")
        print(f"  Conflict: {n_conflict} cặp ({pct:.1f}%)")

    # Color mismatch cases
    color_cases = df_valid_meta[df_valid_meta['color_mismatch'] == True]
    if len(color_cases) > 0:
        n_color_conflict = color_cases['is_conflict'].sum()
        print(f"\nColor mismatch cases ({len(color_cases)} cặp):")
        print(f"  Conflict: {n_color_conflict} ({n_color_conflict/len(color_cases)*100:.1f}%)")

except Exception as e:
    print(f"⚠️  Không load được Pipeline Metadata: {e}")
    print("Bỏ qua phân tích theo tier, tiếp tục bước sau.")
    df_valid_meta = df_valid.copy()
    df_valid_meta['is_conflict'] = df_valid_meta['label_A'] != df_valid_meta['label_B']

CONFLICT ANALYSIS BY TIER

T1_high (80 cặp):
  Conflict: 19 cặp (23.8%)

T2_edge (100 cặp):
  Conflict: 8 cặp (8.0%)

T3_random (20 cặp):
  Conflict: 2 cặp (10.0%)

Color mismatch cases (56 cặp):
  Conflict: 2 (3.6%)


## Bước 6 — Xuất file conflict để review

In [22]:
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

OUTPUT_FILE = 'iaa_results.xlsx'
wb = Workbook()

HEADER_FILL   = PatternFill('solid', fgColor='1F4E79')
HEADER_FONT   = Font(color='FFFFFF', bold=True)
CONFLICT_FILL = PatternFill('solid', fgColor='FFE0E0')
AGREE_FILL    = PatternFill('solid', fgColor='E2EFDA')
TOP_LEFT      = Alignment(horizontal='left', vertical='top', wrap_text=True)
CENTER        = Alignment(horizontal='center', vertical='top')

# ─── Sheet 1: Summary ───────────────────────────────────────
ws1 = wb.active
ws1.title = 'Summary'

summary_rows = [
    ['IAA RESULTS SUMMARY', ''],
    ['', ''],
    ['Metric', 'Giá trị'],
    ['Số cặp đánh giá', n],
    ['Số cặp đồng thuận', agree],
    ['Percent Agreement', f"{pct_agree:.1%}"],
    ['Cohen\'s Kappa', f"{kappa:.3f}"],
    ['Đánh giá', interpret_kappa(kappa)],
    ['', ''],
    ['PHÂN BỔ NHÃN', ''],
    ['Nhãn', 'Annotator A', 'Annotator B'],
]
for label in label_order:
    count_a = labels_A.count(label)
    count_b = labels_B.count(label)
    summary_rows.append([label, count_a, count_b])

summary_rows += [
    ['', ''],
    ['CONFLICT MATRIX (hàng=A, cột=B)', '', 'exact', 'partial', 'irrelevant'],
]
for i, label in enumerate(label_order):
    summary_rows.append([f'A={label}', ''] + cm[i].tolist())

for row in summary_rows:
    ws1.append(row)

ws1.cell(1, 1).font = Font(bold=True, size=14)
ws1.cell(3, 1).fill = HEADER_FILL
ws1.cell(3, 1).font = HEADER_FONT
ws1.cell(3, 2).fill = HEADER_FILL
ws1.cell(3, 2).font = HEADER_FONT
ws1.column_dimensions['A'].width = 28
ws1.column_dimensions['B'].width = 18
ws1.column_dimensions['C'].width = 12
ws1.column_dimensions['D'].width = 12
ws1.column_dimensions['E'].width = 14

# ─── Sheet 2: All Pairs (merged) ────────────────────────────
ws2 = wb.create_sheet('All Pairs')
cols2 = [ID_COL, 'query', 'product_name', 'category', 'type', 'colors',
         'label_A', 'label_B', 'note_A', 'note_B']

# Header
ws2.append(cols2)
for col_idx in range(1, len(cols2)+1):
    ws2.cell(1, col_idx).fill = HEADER_FILL
    ws2.cell(1, col_idx).font = HEADER_FONT
    ws2.cell(1, col_idx).alignment = CENTER

# Data
for _, row in df_valid.iterrows():
    is_conflict = row['label_A'] != row['label_B']
    ws2.append([row.get(c, '') for c in cols2])
    row_fill = CONFLICT_FILL if is_conflict else AGREE_FILL
    for col_idx in range(1, len(cols2)+1):
        cell = ws2.cell(ws2.max_row, col_idx)
        cell.fill = row_fill
        cell.alignment = TOP_LEFT

col_widths2 = [8, 30, 35, 10, 15, 18, 12, 12, 25, 25]
for i, w in enumerate(col_widths2, 1):
    ws2.column_dimensions[get_column_letter(i)].width = w
ws2.freeze_panes = 'A2'

# ─── Sheet 3: Conflicts Only ─────────────────────────────────
ws3 = wb.create_sheet('Conflicts')
cols3 = [ID_COL, 'query', 'product_name', 'category', 'colors',
         'label_A', 'label_B', 'note_A', 'note_B', 'final_label']

ws3.append(cols3)
for col_idx in range(1, len(cols3)+1):
    ws3.cell(1, col_idx).fill = HEADER_FILL
    ws3.cell(1, col_idx).font = HEADER_FONT
    ws3.cell(1, col_idx).alignment = CENTER

df_conflicts = df_valid[df_valid['label_A'] != df_valid['label_B']].copy()
df_conflicts['final_label'] = ''   # cột adjudication — điền sau khi review

for _, row in df_conflicts.iterrows():
    ws3.append([row.get(c, '') for c in cols3])
    for col_idx in range(1, len(cols3)+1):
        cell = ws3.cell(ws3.max_row, col_idx)
        cell.fill = CONFLICT_FILL
        cell.alignment = TOP_LEFT
        if cols3[col_idx-1] == 'final_label':
            cell.fill = PatternFill('solid', fgColor='FFF2CC')

col_widths3 = [8, 30, 35, 10, 18, 12, 12, 25, 25, 15]
for i, w in enumerate(col_widths3, 1):
    ws3.column_dimensions[get_column_letter(i)].width = w
ws3.freeze_panes = 'A2'

wb.save(OUTPUT_FILE)

print(f"  Saved: {OUTPUT_FILE}")
print(f"   Sheet 'Summary'    → tổng hợp kappa, confusion matrix")
print(f"   Sheet 'All Pairs'  → {len(df_valid)} cặp, xanh=đồng thuận, đỏ=conflict")
print(f"   Sheet 'Conflicts'  → {len(df_conflicts)} cặp conflict, cột 'final_label' để adjudicate")

  Saved: iaa_results.xlsx
   Sheet 'Summary'    → tổng hợp kappa, confusion matrix
   Sheet 'All Pairs'  → 200 cặp, xanh=đồng thuận, đỏ=conflict
   Sheet 'Conflicts'  → 29 cặp conflict, cột 'final_label' để adjudicate


## Bước 7 — Download kết quả

In [ ]:
from google.colab import files
files.download(OUTPUT_FILE)
print("✅ Download started!")

---
## Bước 8 — Adjudication (sau khi review Conflicts sheet)

> Upload lại file `iaa_results.xlsx` sau khi đã điền cột `final_label` trong sheet `Conflicts`.
> Cell này tính lại Kappa sau adjudication và xuất tập nhãn cuối cùng.

In [23]:
print("Upload file iaa_results.xlsx đã adjudicate:")
uploaded_adj = files.upload()
FILE_ADJ = list(uploaded_adj.keys())[0]

df_all   = pd.read_excel(FILE_ADJ, sheet_name='All Pairs')
df_conf  = pd.read_excel(FILE_ADJ, sheet_name='Conflicts')
df_conf.columns = df_conf.columns.str.strip()

# Merge final_label vào all pairs
df_conf_slim = df_conf[['ID', 'final_label']].dropna(subset=['final_label'])
df_conf_slim = df_conf_slim[df_conf_slim['final_label'].isin(VALID_LABELS)]

df_final = pd.merge(df_all, df_conf_slim, on='ID', how='left')

# Final label = final_label nếu có conflict, ngược lại dùng label_A
df_final['gold_label'] = df_final.apply(
    lambda r: r['final_label'] if pd.notna(r.get('final_label')) and r['final_label'] in VALID_LABELS
              else r['label_A'], axis=1
)

n_adj = df_conf_slim.shape[0]
n_remain = len(df_conflicts) - n_adj

print(f"\n=== ADJUDICATION SUMMARY ===")
print(f"Tổng conflict:       {len(df_conflicts)}")
print(f"Đã adjudicate:       {n_adj}")
print(f"Chưa adjudicate:     {n_remain}")
print(f"\nPhân bổ gold_label:")
print(df_final['gold_label'].value_counts().to_string())

# Xuất tập nhãn cuối
FINAL_OUTPUT = 'gold_labels.xlsx'
df_final[['ID', 'query', 'product_name', 'category', 'label_A', 'label_B', 'gold_label']].to_excel(
    FINAL_OUTPUT, index=False
)
print(f"\n✅ Saved gold labels: {FINAL_OUTPUT}")
files.download(FINAL_OUTPUT)

Upload file iaa_results.xlsx đã adjudicate:


KeyboardInterrupt: 